<a href="https://colab.research.google.com/github/nabinjoshi54/lis5693/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 2: Text Pre-Processing using spaCy

In this lab, I load my Lab-1 transcript (`transcript.txt`) directly from my GitHub repository which I completed in Lab-1, then use spaCy to perform sentence segmentation, word counts, frequent word analysis, named entity recognition, and phrase matching.

In [1]:
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import requests
import spacy
from collections import Counter

Task 1: Load and read `transcript.txt` from my Lab-1 GitHub repo

In this step, I download my `transcript.txt` file directly from GitHub using a raw file URL and store it as a Python string.

In [3]:
TRANSCRIPT_URL = "https://raw.githubusercontent.com/nabinjoshi54/lis5693/refs/heads/main/lab-1/transcript.txt"

response = requests.get(TRANSCRIPT_URL)
response.raise_for_status()  # stops if the link is wrong
text = response.text

print("Loaded transcript successfully.")

Loaded transcript successfully.


Task 2: Character count and first 100 characters

Here I calculate how many characters are in my transcript and print the first 100 characters to confirm it loaded correctly.

In [4]:
print("Number of characters:", len(text))
print("\nFirst 100 characters:\n")
print(text[:100])

Number of characters: 13171

First 100 characters:

Transcriber: Abdulrahman Sallam
Reviewer: Raúl Higareda
(Applause)
I’m going to ask you
to participa


Task 3: Sentence segmentation using a blank spaCy pipeline

In this step, I create a blank English spaCy pipeline and add the `sentencizer` component to split the transcript into sentences without a full NLP model.

In [5]:
nlp_blank = spacy.blank("en")
nlp_blank.add_pipe("sentencizer")

doc_blank = nlp_blank(text)
sentences = list(doc_blank.sents)

print("Number of sentences:", len(sentences))
print("\nFirst 3 sentences:\n")
for s in sentences[:3]:
    print("-", s.text.strip())

Number of sentences: 146

First 3 sentences:

- Transcriber: Abdulrahman Sallam
Reviewer: Raúl Higareda
(Applause)
I’m going to ask you
to participate in an experiment,
which is that when you leave this room,
when you go out into the world,
today, tomorrow,
or whenever you feel like it,
I’d like you to ask and answer
one question of someone who’s a stranger.
- You might meet them on the bus
or walking down the street.
- I’m going to show you the question
I’m going to ask you to ask and answer.


Task 4: Word count and token analysis

Here I count total words and unique words. I keep only alphabetic tokens and convert them to lowercase for consistent counting.

In [6]:
words = [token.text.lower() for token in doc_blank if token.is_alpha]

print("Total number of words:", len(words))
print("Number of unique words:", len(set(words)))

Total number of words: 2372
Number of unique words: 569


Task 5: Most frequent words

In this step, I find the most frequent words in my transcript. Then I briefly explain why the most frequent word appears often.

In [7]:
word_freq = Counter(words)

print("Top 10 most frequent words:")
for w, c in word_freq.most_common(10):
    print(w, ":", c)

most_common_word, most_common_count = word_freq.most_common(1)[0]
print("\nMost frequent word:", most_common_word)
print("Count:", most_common_count)

Top 10 most frequent words:
to : 92
and : 91
you : 68
the : 58
that : 51
they : 50
a : 46
of : 44
this : 42
i : 38

Most frequent word: to
Count: 92


Why do this word appears frequently?

Answer: The most frequent word is 'to'. This word appears often because it is a common preposition and is repeatedly used in everyday speech and helps connect ideas throughout the transcript. As we can see in the result most repetitive words are 'to', 'and', 'you' etc. and all of them are preposition, conjunction and noun pronous which are critical part of every sentence.

Task 6: Run full spaCy pipeline (Named Entity Recognition)

Here I load the full English spaCy model (`en_core_web_sm`) and extract named entities from the transcript. I report how many entities were found and what entity types appear.

In [8]:
nlp_full = spacy.load("en_core_web_sm")
doc_full = nlp_full(text)

entities = list(doc_full.ents)

print("Number of named entities found:", len(entities))

entity_types = sorted(set([ent.label_ for ent in entities]))
print("\nEntity types found:")
print(entity_types)

Number of named entities found: 46

Entity types found:
['CARDINAL', 'DATE', 'GPE', 'ORDINAL', 'ORG', 'PERSON', 'TIME']


In [9]:
print("\nFirst 20 entities (text -> label):")
for ent in entities[:20]:
    print(ent.text, "->", ent.label_)


First 20 entities (text -> label):
Raúl Higareda -> PERSON
today, tomorrow -> DATE
one -> CARDINAL
A few years ago -> DATE
20 years -> DATE
a long day -> DATE
the New York Times -> ORG
about my day -> DATE
first -> ORDINAL
one -> CARDINAL
one -> CARDINAL
three -> CARDINAL
Behfar Ehdaie -> PERSON
Ehdaie -> PERSON
New York City -> GPE
every single day -> DATE
one -> CARDINAL
Ehdaie -> PERSON
every 6 months -> DATE
every 2 years -> DATE


Task 7: PhraseMatcher

For PhraseMatcher, I selected three phrases related to the topic of my Lab-1 YouTube video and searched for them inside my transcript. The code prints each match and the sentence where it appears.

In [11]:
from spacy.matcher import PhraseMatcher

matcher = PhraseMatcher(nlp_full.vocab, attr="LOWER")

# 3 phrases related to my Lab-1 video topic
phrases = [
    "conversation",
    "communication",
    "data science"
]

patterns = [nlp_full.make_doc(p) for p in phrases]
matcher.add("TOPIC_PHRASES", patterns)

matches = matcher(doc_full)

print("Total phrase matches found:", len(matches))
print("\nMatches:\n")

for match_id, start, end in matches:
    span = doc_full[start:end]
    print("Matched phrase:", span.text)
    print("Sentence:", span.sent.text.strip())
    print("-" * 60)

Total phrase matches found: 19

Matches:

Matched phrase: communication
Sentence: And so, I started talking to researchers
who were studying communication.
------------------------------------------------------------
Matched phrase: communication
Sentence: We’re living through this golden age
of understanding communication,
really for the first time,
because of advances
in neural imaging and data collection.
------------------------------------------------------------
Matched phrase: conversation
Sentence: They said one of the big things
that we’ve learned
Is that we tend to think of a discussion
as being just one conversation.
------------------------------------------------------------
Matched phrase: conversation
Sentence: I was coming home and having
an emotional conversation.
------------------------------------------------------------
Matched phrase: conversation
Sentence: My wife was responding
with a practical conversation.
------------------------------------------------------

Task 8: Reflection

What went well?
Answer: This lab went well because I was able to load my transcript (by copying the raw data link from GitHub) successfully and use spaCy to split sentences, count words, and find frequent words. The spaCy full pipeline also helped me identify named entities and see what types of entities appear in my transcript. Though I created new google colab as instructed in instruction manual in canvas and github, I used the instructions in lab2 (https://github.com/manika-lamba/SP26-LIS4_5693/blob/main/lab-assignments/lab-2/Lab_2.ipynb) to complete lab2.

What did not go well or challenges?
Answer: The first challenge was making sure I used the correct GitHub raw URL for my `transcript.txt` file, which I did not do initially and copied simple text url. After fixing the file path and running cells step-by-step, the tasks worked correctly.